# Probability and Statistics for ML

**Family:** Fundamentals | **Level:** Beginner | **Runs on:** CPU, under a minute

Distributions, sampling error, confidence intervals and a permutation test.

## Learning objectives

- Distinguish population and sample
- Build a bootstrap confidence interval
- Run a permutation test
- Explain why one test split is a noisy estimate

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ml_lab.data import load_dataset
from ml_lab.utils import set_seed
from ml_lab import visualization as viz
SEED = 42
set_seed(SEED)
viz.apply_style()

## 1. Sampling error

We draw samples of different sizes from the Auto MPG column and watch the mean settle. Treating the 392 cars as the population is a simplification for teaching.

In [2]:
mpg = load_dataset("auto_mpg").frame["mpg"].values
rng = np.random.default_rng(SEED)
for n in [5, 20, 80, 320]:
    means = [rng.choice(mpg, n, replace=False).mean() for _ in range(500)]
    print(f"n={n}: std of the sample mean {np.std(means):.3f} (theory sigma/sqrt(n) = {mpg.std() / np.sqrt(n):.3f})")

n=5: std of the sample mean 3.352 (theory sigma/sqrt(n) = 3.486)
n=20: std of the sample mean 1.696 (theory sigma/sqrt(n) = 1.743)
n=80: std of the sample mean 0.764 (theory sigma/sqrt(n) = 0.872)
n=320: std of the sample mean 0.184 (theory sigma/sqrt(n) = 0.436)


## 2. Bootstrap interval for a mean

In [3]:
boot = [rng.choice(mpg, len(mpg), replace=True).mean() for _ in range(2000)]
print("mean mpg:", round(float(mpg.mean()), 2), "| 95% bootstrap interval:", np.round(np.percentile(boot, [2.5, 97.5]), 2))

mean mpg: 23.45 | 95% bootstrap interval: [22.64 24.22]


## 3. Permutation test

Are US cars (origin 1) different in mpg from Japanese cars (origin 3)? We shuffle the group labels many times and see how often a difference this large appears by chance.

In [4]:
df = load_dataset("auto_mpg").frame
g1, g3 = df[df.origin == 1]["mpg"].values, df[df.origin == 3]["mpg"].values
obs = g3.mean() - g1.mean()
pool = np.r_[g1, g3]
diffs = []
for _ in range(5000):
    rng.shuffle(pool)
    diffs.append(pool[len(g1):].mean() - pool[:len(g1)].mean())
print("observed difference:", round(float(obs), 2), "| share of shuffles at least as extreme:", float(np.mean(np.abs(diffs) >= abs(obs))))

observed difference: 10.42 | share of shuffles at least as extreme: 0.0


## 4. Why one test split is noisy

The same model scored on 30 different random splits gives a spread of test scores.

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
X, y = df[["weight", "horsepower", "model_year"]], df["mpg"]
rmses = []
for s in range(30):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=s)
    rmses.append(mean_squared_error(yte, LinearRegression().fit(Xtr, ytr).predict(Xte)) ** 0.5)
print("test RMSE over 30 splits: mean", round(float(np.mean(rmses)), 2), "| std", round(float(np.std(rmses, ddof=1)), 2), "| range", round(min(rmses), 2), "to", round(max(rmses), 2))

test RMSE over 30 splits: mean 3.51 | std 0.3 | range 3.08 to 4.33


## Exercises

1. Repeat the permutation test for mpg of Europe versus Japan and compare p values.
2. Plot the bootstrap distribution of the median.

## Summary

- Sample statistics have spread.
- Bootstrap and permutation tests need few assumptions.
- Report spread, not one number.

**Next:** Calculus and optimisation.

**Data note.** Real datasets come from `ml_lab.data` with licence notes in `datasets/catalog`. Synthetic numbers appear only where a demonstration needs controlled data, and are labelled.